# Data Cleaning — INE Encuesta de Ocupación Hotelera (20/07/2026)

Limpieza de las **4 tablas de la EOH** que sirven de fuente externa para el Sprint 4.

| Tabla | Contenido | Nivel |
|---|---|---|
| 2078 | Viajeros y pernoctaciones por puntos turísticos | Municipio |
| 2039 | Viajeros y pernoctaciones por zonas turísticas | Zona (islas, costas) |
| 2074 | Viajeros y pernoctaciones por CCAA y provincias | Nacional / CCAA / provincia |
| 2069 | Distribución % en cada provincia según CCAA de procedencia | Provincia |

**Criterio**: este notebook corrige formato y marca contenido, **sin eliminar ni una fila**,
igual que el cleaning del dataset de alojamientos. Toda decisión de ámbito (qué mercados,
qué años, qué nivel de la jerarquía) se deja para *Data Transformation*.

Los datos del INE no llegan "sucios" en el sentido habitual: llegan **limpios pero
heterogéneos y con trampas que no dan error, dan un número equivocado**. Los cuatro
ficheros tienen cuatro esquemas distintos (5, 5, 8 y 7 columnas) para la misma información.

In [62]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

In [63]:
# Ruta y sufijo: únicas líneas a cambiar cuando lleguen ficheros nuevos del INE
DATA = Path(r"E:\informacion y documentos\Curso analisis de datos IT Academy"
            r"\Simulador empresarial\ProjecteData\Equip_34\Data")
SUFIJO = "raw_20_07_2026"
TABLAS = ['2078', '2039', '2074', '2069']

In [64]:
# Los CSV del INE son UTF-8 CON BOM. Leerlos como latin-1 corrompe todas las tildes
# ("Autónomas" -> "AutÃ³nomas") y rompe cualquier cruce posterior por nombre.
# Se carga todo como texto: la conversión de tipos se hace explícita en el paso 1.
SUFIJO = "raw_20_07_2026"
TABLAS = ["2078", "2039", "2074", "2069"]

def encontrar_carpeta_data():
    """
    Busca automáticamente la carpeta Data del proyecto.
    Compatible con macOS, Windows y Linux.
    """
    ubicacion_actual = Path.cwd()

    candidatos = [
        ubicacion_actual / "Data",
        ubicacion_actual.parent / "Data",
        ubicacion_actual / "Equip_34" / "Data",
        ubicacion_actual.parent / "Equip_34" / "Data",
    ]

    for carpeta in candidatos:
        if carpeta.exists() and carpeta.is_dir():
            return carpeta

    raise FileNotFoundError(
        "No se encontró la carpeta Data. "
        "Ejecuta el notebook desde la carpeta Equip_34 o Scripts."
    )

DATA = encontrar_carpeta_data()

El BOM se pega al nombre de la primera columna (`﻿Totales Territoriales`). Al usar
`utf-8-sig` pandas lo retira solo; se comprueba abajo que ningún nombre de columna arrastra
caracteres invisibles.

## 1. Corrección de tipos de datos

`Total` llega como texto en **formato europeo**: punto de miles y coma decimal
(`12.074.455`, `2,73`). `Periodo` llega como `2025M08`.

In [65]:
raw = {}

for t in TABLAS:
    archivo = DATA / f"EOH_{t}_{SUFIJO}.csv"

    raw[t] = pd.read_csv(
        archivo,
        sep=";",
        encoding="utf-8-sig",
        dtype=str
    )

print("Archivos cargados correctamente.")

Archivos cargados correctamente.


In [66]:
def a_numero(serie):
    """'12.074.455' -> 12074455.0 ; '2,73' -> 2.73

    Formato europeo: el punto es separador de miles y la coma es decimal. Hay que
    quitar los puntos ANTES de cambiar la coma, o '2,73' se convertiría en '273'.
    """
    return pd.to_numeric(
        serie.str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce')


for t, df in raw.items():
    df['valor'] = a_numero(df['Total'])
    df['anio'] = df['Periodo'].str[:4].astype(int)
    df['mes'] = df['Periodo'].str[5:].astype(int)
    # fecha real al día 1 del mes, para poder ordenar y graficar series temporales
    df['fecha'] = pd.to_datetime(dict(year=df['anio'], month=df['mes'], day=1))

print(raw['2074'][['Periodo', 'anio', 'mes', 'fecha', 'Total', 'valor']].head(3).to_string(index=False))

Periodo  anio  mes      fecha      Total      valor
2026M05  2026    5 2026-05-01 12.074.455 12074455.0
2026M04  2026    4 2026-04-01 10.211.051 10211051.0
2026M03  2026    3 2026-03-01  8.270.160  8270160.0


## 2. Distinción entre "sin dato" y cero

El INE marca con `.` las celdas **sin dato publicado**: secreto estadístico, o ningún
establecimiento de la muestra abierto ese mes. **No son ceros.**

Es el paso más importante de todo el notebook. `pd.to_numeric(errors='coerce')` las
convierte en `NaN`, y a partir de ahí `groupby().sum()` devuelve **0**, fabricando ceros
que no existen: totales anuales infraestimados y "temporadas cerradas" que el dato no
afirma en ningún momento. Se marcan con un flag para que la fase de transformación pueda
agregar con `min_count=1`.

In [67]:
MARCAS_SIN_DATO = ['.', '..']

for t, df in raw.items():
    df['sin_dato'] = df['Total'].isin(MARCAS_SIN_DATO) | df['Total'].isna()

resumen = pd.DataFrame({
    'filas': {t: len(df) for t, df in raw.items()},
    'celdas sin dato': {t: int(df['sin_dato'].sum()) for t, df in raw.items()},
})
resumen['% sin dato'] = (resumen['celdas sin dato'] / resumen['filas'] * 100).round(2)
print(resumen.to_string())

       filas  celdas sin dato  % sin dato
2078  141864            44996       31.72
2039   63168            13876       21.97
2074  138180               85        0.06
2069  697480            40631        5.83


In [68]:
# Verificación: 'sin_dato' debe coincidir exactamente con los NaN de 'valor'.
# Si no coincidiera, habría valores no numéricos distintos de '.' sin detectar.
for t, df in raw.items():
    desajuste = int((df['sin_dato'] != df['valor'].isna()).sum())
    otros = sorted(set(df.loc[df['valor'].isna() & ~df['Total'].isin(MARCAS_SIN_DATO), 'Total'].dropna()))
    print(f"{t}: desajustes = {desajuste} | marcas no previstas = {otros if otros else 'ninguna'}")

2078: desajustes = 0 | marcas no previstas = ninguna
2039: desajustes = 0 | marcas no previstas = ninguna
2074: desajustes = 0 | marcas no previstas = ninguna
2069: desajustes = 0 | marcas no previstas = ninguna


## 3. Marcado de la jerarquía y unificación de los cuatro esquemas

Las tablas del INE son **jerárquicas**: en las mismas columnas conviven la fila del total y
las de su desglose. En la 2074 hay tres jerarquías simultáneas:

- `Provincias` vacío → la fila es el **total de la comunidad**
- `Comunidades y Ciudades Autónomas` vacío → la fila es el **total nacional**
- `Residencia: Nivel 2` vacío → la fila es el **total de residencia** (España + extranjero)

Sumar sin filtrar da **exactamente el doble**. Como el cleaning no elimina filas, se añade
`nivel_jerarquico` (`total` / `desglose`) y `nivel_geo` para que la fase de transformación
elija, y para que nadie sume ambos niveles por accidente.

Aprovechando el paso, los cuatro esquemas se normalizan a uno común en formato largo.

In [69]:
ESQUEMA = ['tabla', 'nivel_geo', 'geo_raw', 'dimension', 'categoria_raw', 'metrica',
           'nivel_jerarquico', 'periodo', 'anio', 'mes', 'fecha', 'valor', 'sin_dato']


def normaliza_geografica(df, tabla, nivel_geo, col_geo):
    """2078 y 2039: una columna geográfica, una de residencia. Sin filas de total."""
    return pd.DataFrame({
        'tabla': tabla, 'nivel_geo': nivel_geo, 'geo_raw': df[col_geo],
        'dimension': 'residencia', 'categoria_raw': df['Residencia'],
        'metrica': df['Viajeros y pernoctaciones'],
        'nivel_jerarquico': 'desglose',
        'periodo': df['Periodo'], 'anio': df['anio'], 'mes': df['mes'], 'fecha': df['fecha'],
        'valor': df['valor'], 'sin_dato': df['sin_dato']})[ESQUEMA]


def normaliza_2074(df):
    """Nacional / CCAA / provincia según qué columnas vengan vacías."""
    ccaa, prov = df['Comunidades y Ciudades Autónomas'], df['Provincias']
    return pd.DataFrame({
        'tabla': '2074',
        'nivel_geo': np.select([prov.notna(), ccaa.notna()], ['provincia', 'ccaa'], 'nacional'),
        'geo_raw': prov.fillna(ccaa).fillna('Total Nacional'),
        'dimension': 'residencia',
        'categoria_raw': df['Residencia: Nivel 2'].fillna('Total'),
        'metrica': df['Viajeros y pernoctaciones'],
        'nivel_jerarquico': np.where(df['Residencia: Nivel 2'].notna(), 'desglose', 'total'),
        'periodo': df['Periodo'], 'anio': df['anio'], 'mes': df['mes'], 'fecha': df['fecha'],
        'valor': df['valor'], 'sin_dato': df['sin_dato']})[ESQUEMA]


def normaliza_2069(df):
    """Provincia de destino x comunidad de procedencia. Los valores son PORCENTAJES."""
    prov = df['Provincias']
    return pd.DataFrame({
        'tabla': '2069',
        'nivel_geo': np.where(prov.notna(), 'provincia', 'nacional'),
        'geo_raw': prov.fillna('Total Nacional'),
        'dimension': 'ccaa_procedencia',
        'categoria_raw': df['Comunidades y Ciudades Autónomas'].fillna('Total'),
        'metrica': df['Viajeros y pernoctaciones'],
        'nivel_jerarquico': np.where(df['Comunidades y Ciudades Autónomas'].notna(),
                                     'desglose', 'total'),
        'periodo': df['Periodo'], 'anio': df['anio'], 'mes': df['mes'], 'fecha': df['fecha'],
        'valor': df['valor'], 'sin_dato': df['sin_dato']})[ESQUEMA]


eoh = pd.concat([
    normaliza_geografica(raw['2078'], '2078', 'punto_turistico', 'Puntos turísticos'),
    normaliza_geografica(raw['2039'], '2039', 'zona_turistica', 'Zonas Turísticas'),
    normaliza_2074(raw['2074']),
    normaliza_2069(raw['2069']),
], ignore_index=True)

print(f"Filas unificadas: {len(eoh):,}")

display(eoh.groupby(["tabla", "nivel_geo", "nivel_jerarquico"]).size().rename("filas").reset_index())

Filas unificadas: 1,040,692


,tabla,nivel_geo,nivel_jerarquico,filas
0,2039,zona_turistica,desglose,63168
1,2069,nacional,desglose,12502
2,2069,nacional,total,658
3,2069,provincia,desglose,650104
4,2069,provincia,total,34216
5,2074,ccaa,desglose,25004
6,2074,ccaa,total,12502
7,2074,nacional,desglose,1316
8,2074,nacional,total,658
9,2074,provincia,desglose,65800


In [70]:
# Verificación crítica: NO se ha perdido ni duplicado ninguna fila
filas_origen = sum(len(df) for df in raw.values())

assert len(eoh) == filas_origen, ("El número de filas unificadas no coincide con el origen.")

print(f"OK: se conservan las {len(eoh):,} filas de origen.")

OK: se conservan las 1,040,692 filas de origen.


## 4. Separación de código INE y nombre geográfico

El INE mete **dos datos en una celda**: `08019 Barcelona`, `01 Andalucía`,
`Baleares (Illes): Isla De Mallorca`. Se separan en `codigo_ine` y `geo`, porque el código
es la clave estable (el nombre cambia de grafía entre tablas) y el nombre es lo legible.

Las zonas turísticas no llevan código numérico sino prefijo de comunidad
(`Cataluña: Costa Brava`); se conserva el nombre completo y se extrae la comunidad.

In [71]:
PATRON_CODIGO = r'^(\d{2,5})\s+(.*)$'   # '08019 Barcelona' / '01 Andalucía'

extraido = eoh['geo_raw'].str.extract(PATRON_CODIGO)
eoh['codigo_ine'] = extraido[0]
eoh['geo'] = extraido[1].fillna(eoh['geo_raw'])

# En zonas turísticas el prefijo es la comunidad, no un código: 'Cataluña: Costa Brava'
es_zona = eoh['nivel_geo'] == 'zona_turistica'
eoh.loc[es_zona, 'geo'] = eoh.loc[es_zona, 'geo_raw'].str.split(':').str[-1].str.strip()
eoh['zona_ccaa'] = np.where(es_zona, eoh['geo_raw'].str.split(':').str[0].str.strip(), np.nan)

print(eoh.loc[eoh['codigo_ine'].notna(), ['geo_raw', 'codigo_ine', 'geo']].drop_duplicates().head(4).to_string(index=False))
print()
print(eoh.loc[es_zona, ['geo_raw', 'zona_ccaa', 'geo']].drop_duplicates().head(4).to_string(index=False))

               geo_raw codigo_ine              geo
 01059 Vitoria-Gasteiz      01059  Vitoria-Gasteiz
        02003 Albacete      02003         Albacete
03014 Alacant/Alicante      03014 Alacant/Alicante
           03018 Altea      03018            Altea

                            geo_raw zona_ccaa                      geo
        Andalucía: Costa De Almería Andalucía         Costa De Almería
Andalucía: Costa De La Luz De Cádiz Andalucía Costa De La Luz De Cádiz
Andalucía: Costa Tropical (Granada) Andalucía Costa Tropical (Granada)
Andalucía: Costa De La Luz (Huelva) Andalucía Costa De La Luz (Huelva)


## 5. Unificación de sinónimos geográficos

El INE escribe el mismo ente con grafías distintas según la tabla —el equivalente
geográfico de los sinónimos de `amenities_list` (Wifi ↔ Wireless Internet). Casos típicos:
`Valencia` / `València`, o los dobles topónimos `Alicante/Alacant`, `Castellón/Castelló`,
`Araba/Álava`.

Se crea `geo_key` en ASCII sin tildes como **clave de cruce**, conservando `geo` con la
grafía original para mostrar.

**Ojo con el código INE**: los códigos de comunidad y de provincia **comparten
numeración**. `01` es Andalucía como comunidad y Araba/Álava como provincia; `04` es
Balears como comunidad y Almería como provincia. Cruzar por `codigo_ine` a secas mezcla
ambos: la clave real es **`(nivel_geo, codigo_ine)`**.

In [72]:
def a_clave(s):
    """Quita tildes y pasa a minúsculas: 'València' y 'Valencia' -> 'valencia'."""
    if not isinstance(s, str):
        return s
    return unicodedata.normalize('NFKD', s.strip()).encode('ascii', 'ignore').decode().lower()


eoh['geo_key'] = eoh['geo'].map(a_clave)
eoh['categoria'] = eoh['categoria_raw'].str.replace(r'^\d{2}\s+', '', regex=True)
eoh['categoria_key'] = eoh['categoria'].map(a_clave)

# Detección de sinónimos: un mismo código con más de una grafía DENTRO de su nivel.
# Agrupar solo por codigo_ine daría 19 falsos positivos por la colisión CCAA/provincia.
dobles = (eoh[eoh['codigo_ine'].notna()]
          .groupby(['nivel_geo', 'codigo_ine'])['geo'].nunique().pipe(lambda s: s[s > 1]))
print(f"códigos con más de una grafía dentro de su nivel: {len(dobles)}")
for (niv, cod) in dobles.index[:8]:
    sel = (eoh['nivel_geo'] == niv) & (eoh['codigo_ine'] == cod)
    print(f"  {niv} {cod}: {sorted(eoh.loc[sel, 'geo'].unique())}"
          f"  ->  geo_key: {sorted(eoh.loc[sel, 'geo_key'].unique())}")

códigos con más de una grafía dentro de su nivel: 0


In [73]:
# Demostración de la colisión de numeración entre niveles (por qué la clave es doble)
colision = (eoh[eoh['codigo_ine'].isin(['01', '04', '07'])]
            .groupby(['codigo_ine', 'nivel_geo'])['geo'].first().unstack())
print(colision.to_string())

nivel_geo              ccaa       provincia
codigo_ine                                 
01                Andalucía     Araba/Álava
04           Balears, Illes         Almería
07          Castilla y León  Balears, Illes


## 6. Flags de calidad del dato

Tres avisos que el INE publica en la ficha de las tablas y que no viajan dentro del CSV.
Se incorporan como columnas para que no se pierdan por el camino:

- **`unidad`** — la 2069 son **porcentajes**, no cuentas. Mezclarla con las demás en una
  misma medida da totales sin sentido.
- **`provisional`** — *"los datos de junio de 2025 y posteriores son provisionales"*.
- **`serie_enlazada`** — *"debido a distintas actualizaciones en los directorios de
  establecimientos, no son directamente comparables los datos de distintos años"*. Hay
  coeficientes de enlace hasta **febrero de 2012**; a partir de marzo de 2012 la serie es
  homogénea sin aplicarlos.

In [74]:
eoh['metrica'] = eoh['metrica'].str.lower().replace({'viajero': 'viajeros'})

eoh['unidad'] = np.where(eoh['tabla'] == '2069', 'porcentaje',
                         np.where(eoh['metrica'] == 'pernoctaciones', 'noches', 'personas'))

eoh['provisional'] = eoh['fecha'] >= '2025-06-01'
eoh['serie_enlazada'] = eoh['fecha'] <= '2012-02-01'

print(eoh.groupby(['unidad', 'metrica']).size().rename('filas').to_string())
print()
print(f"filas provisionales   : {eoh['provisional'].sum():>9,}")
print(f"filas con enlace serie: {eoh['serie_enlazada'].sum():>9,}")

unidad      metrica       
noches      pernoctaciones    171606
personas    viajeros          171606
porcentaje  pernoctaciones    348740
            viajeros          348740

filas provisionales   :    39,408
filas con enlace serie:   479,128


## 7. Validación

Cinco comprobaciones antes de exportar: que no se han perdido filas, que no hay duplicados,
que los importes cuadran con el origen, que la jerarquía está bien marcada (el total debe ser
la suma de su desglose) y que los porcentajes de la 2069 suman ~100. Son los controles de
calidad que pide la justificación técnica del sprint.

In [75]:
# 7.1 Duplicados: cada combinación geografía × dimensión × métrica × periodo debe ser única.
# La justificación técnica del sprint pide comprobar duplicados por destino, periodo,
# residencia e indicador antes de integrar; esta es la clave equivalente en formato largo.
CLAVE = ['tabla', 'nivel_geo', 'geo_raw', 'dimension', 'categoria_raw', 'metrica', 'periodo']
dups = eoh.duplicated(subset=CLAVE, keep=False)
print(f"filas duplicadas por {CLAVE}: {int(dups.sum())}")
assert dups.sum() == 0, "hay filas duplicadas: revisar la clave antes de exportar"
print("OK: ninguna combinación se repite")

filas duplicadas por ['tabla', 'nivel_geo', 'geo_raw', 'dimension', 'categoria_raw', 'metrica', 'periodo']: 0
OK: ninguna combinación se repite


In [76]:
# 7.2 Importes: la suma de 'valor' por tabla debe coincidir con el fichero de origen
for t in TABLAS:
    o = a_numero(raw[t]['Total']).sum()
    c = eoh.loc[eoh['tabla'] == t, 'valor'].sum()
    print(f"{t}: origen {o:>18,.2f} | limpio {c:>18,.2f} | dif {o - c:.4f}")

2078: origen   5,212,754,667.00 | limpio   5,212,754,667.00 | dif 0.0000
2039: origen   8,318,473,451.00 | limpio   8,318,473,451.00 | dif 0.0000
2074: origen  58,932,880,015.00 | limpio  58,932,880,015.00 | dif 0.0000
2069: origen       6,894,607.77 | limpio       6,894,607.77 | dif 0.0000


In [77]:
# 7.3 Jerarquía: en la 2074 el 'total' de residencia debe ser la suma de su desglose.
# Diferencias de 1-4 unidades sobre decenas de millones son redondeo del PROPIO INE:
# publica cada agregado estimado y redondeado por separado, no como suma exacta.
# Lo que se valida aquí es que el marcado 'total' / 'desglose' es correcto.
p = eoh[(eoh['tabla'] == '2074') & (eoh['nivel_geo'] == 'ccaa') &
        (eoh['metrica'] == 'pernoctaciones') & (eoh['anio'] == 2025)]
tot = p[p['nivel_jerarquico'] == 'total'].groupby('geo')['valor'].sum(min_count=1)
des = p[p['nivel_jerarquico'] == 'desglose'].groupby('geo')['valor'].sum(min_count=1)
comp = pd.DataFrame({'fila total': tot, 'suma desglose': des})
comp['dif'] = (comp['fila total'] - comp['suma desglose']).abs()
comp['dif %'] = (comp['dif'] / comp['fila total'] * 100).round(6)
print(f"CCAA comparadas: {len(comp)} | dif máxima: {comp['dif'].max():,.0f} "
      f"pernoctaciones ({comp['dif %'].max():.6f}%)")
print(comp.head(5).to_string())

CCAA comparadas: 19 | dif máxima: 4 pernoctaciones (0.001448%)
                         fila total  suma desglose  dif     dif %
geo                                                              
Andalucía                57217615.0     57217614.0  1.0  0.000002
Aragón                    5928553.0      5928552.0  1.0  0.000017
Asturias, Principado de   3849033.0      3849034.0  1.0  0.000026
Balears, Illes           63389010.0     63389010.0  0.0  0.000000
Canarias                 72827378.0     72827379.0  1.0  0.000001


In [78]:
# 7.4 La 2069 son porcentajes: las CCAA de procedencia deben sumar ~100 en cada provincia
q = eoh[(eoh['tabla'] == '2069') & (eoh['nivel_jerarquico'] == 'desglose') &
        (eoh['metrica'] == 'pernoctaciones') & (eoh['periodo'] == '2025M08')]
s = q.groupby('geo')['valor'].sum(min_count=1)
print(f"provincias: {len(s)} | mín {s.min():.2f}% | máx {s.max():.2f}% | fuera de [99,101]: {((s < 99) | (s > 101)).sum()}")

provincias: 53 | mín 99.97% | máx 100.02% | fuera de [99,101]: 0


In [79]:
# 7.5 Panorama final de nulos: TODOS los nulos de 'valor' deben venir de 'sin_dato'
print(eoh[ESQUEMA + ['codigo_ine', 'geo', 'geo_key', 'unidad']].isna().sum().rename('nulos').to_string())
print()
print(f"nulos de 'valor' no explicados por sin_dato: {int((eoh['valor'].isna() & ~eoh['sin_dato']).sum())}")

tabla                   0
nivel_geo               0
geo_raw                 0
dimension               0
categoria_raw           0
metrica                 0
nivel_jerarquico        0
periodo                 0
anio                    0
mes                     0
fecha                   0
valor               99588
sin_dato                0
codigo_ine          78302
geo                     0
geo_key                 0
unidad                  0

nulos de 'valor' no explicados por sin_dato: 0


## 8. Exportación

Se exporta **íntegro**: las 138 zonas de puntos turísticos, las 48 zonas turísticas, las 19
comunidades y las 52 provincias, con toda la serie histórica. Ningún departamento pierde
filas por decisiones tomadas aquí.

El recorte al ámbito de StaySpain (8 mercados, ventana temporal, elección del nivel de la
jerarquía) corresponde a *Data Transformation*.

In [80]:
COLUMNAS_SALIDA = ['tabla', 'nivel_geo', 'codigo_ine', 'geo', 'geo_key', 'zona_ccaa',
                   'dimension', 'categoria', 'categoria_key', 'metrica', 'unidad',
                   'nivel_jerarquico', 'periodo', 'anio', 'mes', 'fecha',
                   'valor', 'sin_dato', 'provisional', 'serie_enlazada']

salida = eoh[COLUMNAS_SALIDA].sort_values(['tabla', 'nivel_geo', 'geo', 'dimension',
                                           'categoria', 'metrica', 'fecha'])
destino = DATA / "clean_dataset_INE_EOH_20_07_2026.csv"
salida.to_csv(destino, index=False, sep=';', decimal=',', encoding='utf-8-sig')

print(f"Exportado: {destino.name}")
print(f"  {len(salida):,} filas x {len(COLUMNAS_SALIDA)} columnas")
print(f"  {destino.stat().st_size / 1048576:.1f} MB")
print(f"  serie {salida['periodo'].min()} .. {salida['periodo'].max()}")

Exportado: clean_dataset_INE_EOH_20_07_2026.csv
  1,040,692 filas x 20 columnas
  168.3 MB
  serie 1999M01 .. 2026M05


In [81]:
salida.head(5)

,tabla,nivel_geo,codigo_ine,geo,geo_key,zona_ccaa,dimension,categoria,categoria_key,metrica,unidad,nivel_jerarquico,periodo,anio,mes,fecha,valor,sin_dato,provisional,serie_enlazada
166538,2039,zona_turistica,NaN,Barcelona,barcelona,Cataluña,residencia,Residentes en España,residentes en espana,pernoctaciones,noches,desglose,1999M01,1999,1,1999-01-01,179154.0,False,False,True
166537,2039,zona_turistica,NaN,Barcelona,barcelona,Cataluña,residencia,Residentes en España,residentes en espana,pernoctaciones,noches,desglose,1999M02,1999,2,1999-02-01,235791.0,False,False,True
166536,2039,zona_turistica,NaN,Barcelona,barcelona,Cataluña,residencia,Residentes en España,residentes en espana,pernoctaciones,noches,desglose,1999M03,1999,3,1999-03-01,239266.0,False,False,True
166535,2039,zona_turistica,NaN,Barcelona,barcelona,Cataluña,residencia,Residentes en España,residentes en espana,pernoctaciones,noches,desglose,1999M04,1999,4,1999-04-01,269701.0,False,False,True
166534,2039,zona_turistica,NaN,Barcelona,barcelona,Cataluña,residencia,Residentes en España,residentes en espana,pernoctaciones,noches,desglose,1999M05,1999,5,1999-05-01,238228.0,False,False,True
